# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
2. **String Cleaning [<u>[click]</u>](#string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words like "dan").
3. **Manual Overrides [<u>[click]</u>](#manual-overrides):** Applied a manual dictionary mapping for 5 persistent edge cases (e.g., "Kepulauan Tanimbar", "Unknown Location") to achieve 100% province mapping with zero null values.
4. **Feature Engineering [<u>[click]</u>](#feature-engineering):** Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`) to mathematically normalize the right-skewed "whale" distributions in applicant and quota counts.
5. **Schema Finalization [<u>[click]</u>](#schema-finalization):** Renamed the `job_location` column to `regency_city` for greater accuracy and reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [1]:
# Import libraries
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [2]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# Data Integration
Mapping province names to their respective regency/city names using the administrative divisions dictionary.

In [3]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [ ]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

In [ ]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [ ]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


# String Cleaning

In [46]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [47]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

# Manual Overrides

In [48]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [49]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

# Feature Engineering

In [51]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
1854,a23f6618-2af9-4915-a758-f7ee9cdca4b4,2026-07-16T12:52:10+07:00,PEMBINAAN KEPRIBADIAN,RUMAH TAHANAN NEGARA KELAS IIB TANJUNG,Kab. Tabalong,Bachelor,"Pendidikan Seni Tari dan Musik, Seni Tari, Pen...",1.\tMenyusun dan melaksanakan program pembinaa...,6,1,1,3,Kalimantan Selatan,25.00
7582,a239538b-9641-4d41-83c2-d112b3a914a3,2026-07-16T09:44:51+07:00,Admin Operasional,Adhitama Mitra Nusantara,Kota Adm. Jakarta Utara,"Bachelor, Diploma","Manajemen, Manajemen Administrasi",- Membantu proses administrasi PIC untuk penge...,5,1,1,5,Dki Jakarta,16.67
9093,a244109a-273c-41c9-9239-9c9d24638c78,2026-07-16T12:17:35+07:00,BPK – Asisten Penyusunan Program OR TKPEKM,Sekretariat Utama,Kota Adm. Jakarta Selatan,"Diploma, Bachelor","Ekonomi Pembangunan, Manajemen, Ilmu Ekonomi, ...",Membantu tim dalam pengelolaan keuangan; pelak...,5,3,3,17,Dki Jakarta,16.67
11136,a2415596-265e-4838-be18-654ea73ae37d,2026-07-16T09:53:16+07:00,Supervisi Elektronika,Pal Indonesia,Kota Surabaya,"Bachelor, Diploma",Elektronika,"Melaksanakan pengawasan pekerjaan Elektronika,...",5,1,1,6,Jawa Timur,14.29
21811,a240fb9e-aac5-4354-a079-a2654017ad85,2026-07-16T11:16:47+07:00,Graphic Designer - IT Digital,Perusahaan Perseroan (Persero) PT. Telekomunik...,Kota Adm. Jakarta Selatan,"Diploma, Bachelor","Sistem Informasi, Ilmu Komunikasi, Desain Komu...",$24,5,1,1,12,Dki Jakarta,7.69
13773,a2417e67-36d0-4ba2-99f9-702f58df2e9b,2026-07-16T10:19:47+07:00,Tenaga Gizi (Ahli Gizi / Dietisien),PT. Adi Multi Husada,Kab. Sleman,"Diploma, Profession","Gizi dan Dietetika, Gizi Klinik, Gizi, Ilmu Gizi",Kurikulum ini dirancang untuk membekali pesert...,6,1,1,7,Di Yogyakarta,12.50
10199,a2434735-8b2b-4bd5-aef0-c840f6eb3060,2026-07-16T11:22:16+07:00,IT Analyst Trainee,Esco Bintan Indonesia,Kab. Bintan,"Bachelor, Diploma",Teknik Informatika,1. Assist in ERP system testing and verificati...,5,1,1,6,Kepulauan Riau,14.29
4786,a241635f-f5d2-41b0-9163-04554e805af5,2026-07-16T10:13:59+07:00,Developer RPA/OCR Intern,Sinergi Informatika Semen Indonesia,Kota Adm. Jakarta Selatan,Bachelor,"Teknik Informatika, Sistem Dan Teknologi Infor...",Membantu mengembangkan dan mengimplementasikan...,5,2,2,8,Dki Jakarta,22.22
14912,a237406c-e511-4065-9d49-4a33e60f5e07,2026-07-16T10:26:29+07:00,Social Media Promotion,PT Mdtv Media Televisi,Kota Adm. Jakarta Selatan,Bachelor,"Multimedia, Ilmu Komunikasi, Pemasaran Digital...","Content Planning social media,\n Copywriting s...",5,2,2,15,Dki Jakarta,12.50
14588,a241a557-33eb-4a3f-a1d8-39315c111a46,2026-07-16T10:25:26+07:00,Social Media Intern,Surya Citra Televisi,Kota Adm. Jakarta Pusat,"Diploma, Bachelor","Multimedia, Ilmu Komunikasi, Desain Komunikasi...","- Merencanakan, membuat, dan mengelola konten ...",5,4,4,29,Dki Jakarta,13.33


# Schema Finalization

In [52]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "company",
    "regency_city",
    "province",
    "education_level",
    "allowed_major",
    "job_description",
    "weekly_working_day",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions.rename(columns={"job_location": "regency_city"})[
    final_cols
]

display(internship_postings.sample(10))

,job_id,published_at,job_title,company,regency_city,province,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,acceptance_percentage
14871,a23347cd-b11a-462e-97d0-f92a97c2b6eb,2026-07-16T11:03:46+07:00,BNI Digital Assistant,Perusahaan Perseroan (Persero) Bank Negara Ind...,Kab. Purworejo,Jawa Tengah,"Diploma, Bachelor, Profession","Analisis Keuangan, Administrasi Keuangan Publi...",Mempelajari customer & channel management al t...,5,2,2,15,12.50
23743,a242afa1-eece-4ee8-98cf-21d7f991b548,2026-07-16T12:39:43+07:00,Pengadministrasi Umum - Penyelenggara Pelatihan,BPVP Bantaeng,Kab. Bantaeng,Sulawesi Selatan,"Diploma, Bachelor","Manajemen, Manajemen Administrasi, Manajemen I...","Membantu penyusunan laporan, melakukan pengars...",5,2,2,28,6.90
23434,a243c75f-3817-43dc-b7b6-0f090892765d,2026-07-16T12:10:26+07:00,PENERJEMAH,KANTOR WILAYAH DIREKTORAT JENDERAL IMIGRASI LA...,Kota Bandar Lampung,Lampung,Bachelor,"Sastra Cina, Sastra Inggris","1. Menerjemahkan dokumen, surat, atau naskah r...",5,1,1,14,6.67
16738,a230b7e5-5af9-433d-8f69-3a0d8a58857b,2026-07-16T10:21:15+07:00,Corporate Human Resources,PT. Utama Pratama Medika ( Rs. Emc Tangerang ),Kota Tangerang Selatan,Banten,Bachelor,Psikologi,1. Memahami end to end proses rekrutmen dan im...,5,3,3,25,11.54
20518,a2412279-8ef3-448c-93d8-651e3d77eaab,2026-07-16T12:09:05+07:00,PSIKOLOG ANAK,LEMBAGA PEMASYARAKATAN KELAS IIB MOJOKERTO,Kota Mojokerto,Jawa Timur,Bachelor,"Psikologi, Psikologi Islam, Psikologi Kristen",1. Melaksanakan asesmen psikologis\n2. Memberi...,5,1,1,11,8.33
5259,a241b6ca-34c3-4f41-aa4f-efd101c8a0ef,2026-07-16T11:22:01+07:00,Back-End Engineer,Erhanesia Digima Mukitama,Kab. Purbalingga,Jawa Tengah,"Diploma, Bachelor, Profession","Teknik Informatika, Ilmu Komputer, Rekayasa Pe...",Mengembangkan dan memelihara sistem sisi serve...,6,2,2,9,20.00
19330,a243d97e-2555-483e-a392-b05cf2c96ab6,2026-07-16T11:00:34+07:00,D3 Keperawatan,Rsu Anwar Medika,Kab. Sidoarjo,Jawa Timur,Diploma,"Ilmu Keperawatan, Keperawatan",Kualifikasi:,6,2,1,10,9.09
8482,a238cd62-a6c8-4392-a007-e09ef8664071,2026-07-16T10:24:07+07:00,Internal Control Over Financial Reporting Intern,PT Citilink Indonesia,Kota Tangerang,Banten,Bachelor,"Keuangan, Manajemen, Ekonomi, Akuntansi",1. Membantu pengelolaan administrasi dan dokum...,5,5,5,27,17.86
23659,a240f233-9195-4fdf-a989-595680b7ecb4,2026-07-16T10:06:12+07:00,Commercial Strategy Intern,Siemens Healthineers Indonesia,Kota Adm. Jakarta Selatan,Dki Jakarta,Bachelor,"Teknik Industri, Ekonomi, Teknik Biomedis, St...",Ruang lingkup tanggung jawab Commercial Strate...,5,1,1,14,6.67
18373,a23999c7-61fa-4c56-8bf0-dafdc78d3637,2026-07-16T11:08:58+07:00,HRGA (Human Resources & General Affairs) Staff,PT. Pralon,Kota Tangerang,Banten,Bachelor,"Administrasi BIsnis, Manajemen, Teknik Industr...",Mendukung pelaksanaan administrasi Human Resou...,5,3,3,27,10.71


In [ ]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 